In [1]:
# --- Notebook Test Engine: injected setup (auto-generated) ---
import os as _os
for _k in ('AWS_ACCESS_KEY_ID','AWS_SECRET_ACCESS_KEY','AWS_SESSION_TOKEN','AWS_CREDENTIAL_EXPIRATION'):
    _os.environ.pop(_k, None)
_os.environ.setdefault('AWS_DEFAULT_REGION', 'us-west-2')
_os.environ.setdefault('AWS_REGION', 'us-west-2')
_nte_role = 'arn:aws:iam::674622101542:role/NotebookTestEngine-ProcessingJobRole'
try:
    from sagemaker.core.helper import session_helper as _sh
    _sh.get_execution_role = lambda *a, **k: _nte_role
except Exception:
    pass
try:
    _nte_cf = _os.environ.get('AWS_SHARED_CREDENTIALS_FILE')
    if _nte_cf and _os.path.exists(_nte_cf):
        import configparser as _cfp, datetime as _dt
        import boto3 as _b3, botocore.session as _bcs
        from botocore.credentials import RefreshableCredentials as _RC
        _nte_prof = _os.environ.get('AWS_PROFILE', 'default')
        def _nte_load():
            _cp = _cfp.ConfigParser(); _cp.read(_nte_cf)
            _s = _cp[_nte_prof]
            _tok = _s.get('aws_session_token')
            if not _tok:
                raise RuntimeError('nte: no session token in creds file')
            return {'access_key': _s['aws_access_key_id'],
                    'secret_key': _s['aws_secret_access_key'],
                    'token': _tok,
                    'expiry_time': (_dt.datetime.now(_dt.timezone.utc)
                                    + _dt.timedelta(minutes=9)).isoformat()}
        _nte_creds = _RC.create_from_metadata(metadata=_nte_load(),
                                              refresh_using=_nte_load,
                                              method='nte-shared-file')
        def _nte_get_credentials(self, *a, **k):
            return _nte_creds
        _bcs.Session.get_credentials = _nte_get_credentials
        _b3.setup_default_session(botocore_session=_bcs.get_session())
        print('nte: refreshable shared-file creds installed')
except Exception as _e:
    print('nte: refreshable-creds shim skipped:', _e)
try:
    from sagemaker.core.resources import ModelPackageGroup as _NteMPG
    _nte_mpg_create = _NteMPG.create
    def _nte_mpg_get_or_create(*_a, **_k):
        try:
            return _nte_mpg_create(*_a, **_k)
        except Exception as _e2:
            if 'already exists' in str(_e2).lower():
                _name = _k.get('model_package_group_name') or (_a[0] if _a else None)
                return _NteMPG.get(_name)
            raise
    _NteMPG.create = staticmethod(_nte_mpg_get_or_create)
    print('nte: ModelPackageGroup.create is now get-or-create')
except Exception as _e:
    print('nte: ModelPackageGroup idempotency shim skipped:', _e)


nte: ModelPackageGroup.create is now get-or-create


# SageMaker InspectAI Evaluation

This notebook demonstrates how to use the `InspectAIEvaluator` to run [InspectAI](https://inspect.ai-safety-institute.org.uk/) evaluation tasks on SageMaker infrastructure.

InspectAI is an open-source framework for evaluating LLMs with a broad set of benchmarks and methodologies. The `InspectAIEvaluator` runs InspectAI tasks inside a dedicated container on SageMaker Training, supporting Bedrock inference, existing SageMaker endpoints, or auto-created endpoints.

## Setup

Import necessary modules and configure logging.

In [2]:
# Configure AWS region and credentials via: aws configure, IAM Identity Center (aws sso login), or environment variables.
#! aws configure set region us-east-1

In [3]:
from sagemaker.train.evaluate import InspectAIEvaluator
from rich.pretty import pprint

import logging
logging.basicConfig(
    level=logging.INFO,
    format='%(levelname)s - %(name)s - %(message)s'
)

sagemaker.config INFO - Not applying SDK defaults from location: /etc/xdg/sagemaker/config.yaml


sagemaker.config INFO - Fetched defaults config from location: /opt/ml/processing/input/sm_config.yaml


## Step 1: Prepare Benchmarks

InspectAI benchmarks are Python files containing `@task` decorated functions. Each task defines a dataset, solver, and scorer.

Example benchmark file (`boolq_pt.py`):
```python
from inspect_ai import task, Task
from inspect_ai.dataset import json_dataset
from inspect_ai.scorer import choice
from inspect_ai.solver import multiple_choice

@task
def boolq_pt():
    return Task(
        dataset=json_dataset("boolq_data.json"),
        solver=[multiple_choice()],
        scorer=choice(),
    )
```

Your benchmark directory should also include a `pyproject.toml` and any data files referenced by the task.

### Option A: Use benchmarks already in S3

If your benchmark files are already uploaded to S3, just point to them:

In [4]:
# Point to existing benchmarks in S3
BENCHMARKS_PATH = "s3://notebook-test-engine-ds-674622101542-usw2/inspectai/benchmarks/boolq/"

### Option B: Upload local benchmarks to S3

Use `upload_benchmarks()` to upload a local directory of InspectAI task files.

In [5]:
# Upload local benchmarks to S3
evaluator = InspectAIEvaluator(
    model="nova-textgeneration-lite",
    bedrock_model_id="us.amazon.nova-lite-v1:0",
    benchmarks_path="s3://notebook-test-engine-ds-674622101542-usw2/benchmarks/",  # Will be overwritten
    s3_output_path="s3://notebook-test-engine-ds-674622101542-usw2/output/inspectai-eval/",
)
BENCHMARKS_PATH = "s3://notebook-test-engine-ds-674622101542-usw2/inspectai/benchmarks/boolq/"
print(f"Benchmarks uploaded to: {BENCHMARKS_PATH}")

[07/17/26 03:03:47] INFO     Runs on sagemaker prod, region:us-west-2                                  ]8;id=5602639;file:///usr/local/lib/python3.12/dist-packages/sagemaker/core/utils/utils.py\utils.py]8;;\:]8;id=5602640;file:///usr/local/lib/python3.12/dist-packages/sagemaker/core/utils/utils.py#375\375]8;;\

                    INFO     Found 1 MLflow apps: [('finetune-mlflow-1783049203', 'Created',  ]8;id=5602647;file:///usr/local/lib/python3.12/dist-packages/sagemaker/train/common_utils/finetune_utils.py\finetune_utils.py]8;;\:]8;id=5602648;file:///usr/local/lib/python3.12/dist-packages/sagemaker/train/common_utils/finetune_utils.py#213\213]8;;\
                             '3.10.1')]                                                                            

[07/17/26 03:03:48] INFO     Resolved MLflow app:                                             ]8;id=5602654;file:///usr/local/lib/python3.12/dist-packages/sagemaker/train/common_utils/finetune_utils.py\finetune_utils.py]8;;\:]8;id=5602655;file:///usr/local/lib/python3.12/dist-packages/sagemaker/train/common_utils/finetune_utils.py#236\236]8;;\
                             arn:aws:sagemaker:us-west-2:674622101542:mlflow-app/app-3THB6MFI                      
                             DE4F (status: Created, version: 3.10.1)                                               

                    INFO     Resolved MLflow resource ARN:                                    ]8;id=5602662;file:///usr/local/lib/python3.12/dist-packages/sagemaker/train/evaluate/base_evaluator.py\base_evaluator.py]8;;\:]8;id=5602663;file:///usr/local/lib/python3.12/dist-packages/sagemaker/train/evaluate/base_evaluator.py#215\215]8;;\
                             arn:aws:sagemaker:us-west-2:674622101542:mlflow-app/app-3THB6MFI                      
                             DE4F                                                                                  

Benchmarks uploaded to: s3://notebook-test-engine-ds-674622101542-usw2/inspectai/benchmarks/boolq/


## Step 2: Create InspectAIEvaluator

Create an evaluator instance. The evaluator supports three inference modes:
1. **Bedrock** (default) — use `bedrock_model_id`
2. **Existing SageMaker endpoint** — use `endpoint_name`
3. **Create new endpoint** — use `model_s3_uri` + `inference_image_uri`

In [6]:
# Configuration
REGION = "us-west-2"
S3_OUTPUT_PATH = "s3://notebook-test-engine-ds-674622101542-usw2/output/inspectai-eval/"

In [7]:
# Create evaluator with Bedrock inference
evaluator = InspectAIEvaluator(
    model="nova-textgeneration-lite",
    bedrock_model_id="us.amazon.nova-lite-v1:0",
    benchmarks_path=BENCHMARKS_PATH,
    tasks=[{"name": "boolq_pt", "limit": 10}],
    s3_output_path=S3_OUTPUT_PATH,
    instance_type="ml.m5.large",
    region=REGION,
)

print("InspectAIEvaluator created successfully")
pprint(evaluator)

                    INFO     Found 1 MLflow apps: [('finetune-mlflow-1783049203', 'Created',  ]8;id=5602668;file:///usr/local/lib/python3.12/dist-packages/sagemaker/train/common_utils/finetune_utils.py\finetune_utils.py]8;;\:]8;id=5602669;file:///usr/local/lib/python3.12/dist-packages/sagemaker/train/common_utils/finetune_utils.py#213\213]8;;\
                             '3.10.1')]                                                                            

[07/17/26 03:03:49] INFO     Resolved MLflow app:                                             ]8;id=5602674;file:///usr/local/lib/python3.12/dist-packages/sagemaker/train/common_utils/finetune_utils.py\finetune_utils.py]8;;\:]8;id=5602675;file:///usr/local/lib/python3.12/dist-packages/sagemaker/train/common_utils/finetune_utils.py#236\236]8;;\
                             arn:aws:sagemaker:us-west-2:674622101542:mlflow-app/app-3THB6MFI                      
                             DE4F (status: Created, version: 3.10.1)                                               

                    INFO     Resolved MLflow resource ARN:                                    ]8;id=5602680;file:///usr/local/lib/python3.12/dist-packages/sagemaker/train/evaluate/base_evaluator.py\base_evaluator.py]8;;\:]8;id=5602681;file:///usr/local/lib/python3.12/dist-packages/sagemaker/train/evaluate/base_evaluator.py#215\215]8;;\
                             arn:aws:sagemaker:us-west-2:674622101542:mlflow-app/app-3THB6MFI                      
                             DE4F                                                                                  

InspectAIEvaluator created successfully


InspectAIEvaluator(
│   region='us-west-2',
│   role=None,
│   sagemaker_session=<sagemaker.core.helper.session_helper.Session object at 0x7f4fe3f8fb30>,
│   model='nova-textgeneration-lite',
│   base_model_name=None,
│   base_eval_name='eval-nova-1c30ebe7',
│   s3_output_path='s3://notebook-test-engine-ds-674622101542-usw2/output/inspectai-eval/',
│   mlflow_resource_arn='arn:aws:sagemaker:us-west-2:674622101542:mlflow-app/app-3THB6MFIDE4F',
│   mlflow_experiment_name=None,
│   mlflow_run_name=None,
│   networking=None,
│   kms_key_id=None,
│   model_package_group=None,
│   compute=None,
│   training_image=None,
│   recipe=None,
│   overrides=None,
│   benchmarks_path='s3://notebook-test-engine-ds-674622101542-usw2/inspectai/benchmarks/boolq/',
│   tasks=[{'name': 'boolq_pt', 'limit': 10}],
│   output_format=None,
│   bedrock_model_id='us.amazon.nova-lite-v1:0',
│   endpoint_name=None,
│   model_s3_uri=None,
│   inference_image_uri=None,
│   endpoint_instance_type=None,
│   endpoint_instance_count=1,
│   endpoint_execution_role_arn=None,
│   context_length=None,
│   max_concurrency=None,
│   cleanup_endpoint=True,
│   endpoint_prefix='inspectai',
│   endpoint_environment=None,
│   extra_args=None,
│   environment=None,
│   image_uri=None,
│   instance_type='ml.m5.large',
│   max_runtime_seconds=86400,
│   max_connections=16,
│   max_retries=100,
│   timeout=600,
│   temperature=0.0,
│   top_p=1.0,
│   top_k=-1,
│   max_tokens=8192
)

### Task Configuration Options

The `tasks` parameter accepts a list of dicts with:
- `name` (required): Task function name
- `path` (optional): Path to `.py` file within benchmarks directory
- `limit` (optional): Max samples to evaluate
- `epochs` (optional): Number of evaluation epochs
- `task_args` (optional): Dict of additional task arguments

If `tasks` is omitted, all tasks found in `benchmarks_path` are run.

### Alternative: Using an Existing SageMaker Endpoint

In [8]:
# # Evaluate using an existing SageMaker endpoint
# evaluator = InspectAIEvaluator(
#     model="nova-textgeneration-lite",
#     endpoint_name="<your-endpoint-name>",
#     benchmarks_path=BENCHMARKS_PATH,
#     tasks=[{"name": "boolq_pt"}],
#     s3_output_path=S3_OUTPUT_PATH,
#     role="arn:aws:iam::<account-id>:role/<your-execution-role>",
#     region=REGION,
# )

### Alternative: Auto-Create a New Endpoint

In [9]:
# # Evaluate by creating a new endpoint from model artifacts
# evaluator = InspectAIEvaluator(
#     model="my-model",
#     model_s3_uri="s3://notebook-test-engine-ds-674622101542-usw2/model-artifacts/model.tar.gz",
#     inference_image_uri="<account-id>.dkr.ecr.<region>.amazonaws.com/<image-name>:<tag>",
#     endpoint_instance_type="ml.g5.xlarge",
#     benchmarks_path=BENCHMARKS_PATH,
#     tasks=[{"name": "boolq_pt"}],
#     s3_output_path=S3_OUTPUT_PATH,
#     cleanup_endpoint=True,  # Delete endpoint after evaluation
#     region=REGION,
# )

## Step 3: Tune Decoding Parameters (Optional)

Adjust generation parameters before running the evaluation.

In [10]:
# View current decoding settings
print(f"Temperature: {evaluator.temperature}")
print(f"Top-p: {evaluator.top_p}")
print(f"Top-k: {evaluator.top_k}")
print(f"Max tokens: {evaluator.max_tokens}")
print(f"Max connections: {evaluator.max_connections}")
print(f"Timeout: {evaluator.timeout}s")

Temperature: 0.0
Top-p: 1.0
Top-k: -1
Max tokens: 8192
Max connections: 16
Timeout: 600s


## Step 4: Run Evaluation

Call `evaluate()` to start the InspectAI evaluation job. This will:
1. Build a YAML configuration file
2. Upload it to S3
3. Launch a SageMaker Pipeline with the InspectAI container

In [11]:
# Start evaluation
execution = evaluator.evaluate()

print(f"\n✓ Evaluation started!")
print(f"  Execution ARN: {execution.arn}")
print(f"  Status: {execution.status.overall_status}")

                    INFO     SageMaker Python SDK will collect telemetry to help us better ]8;id=5602688;file:///usr/local/lib/python3.12/dist-packages/sagemaker/core/telemetry/telemetry_logging.py\telemetry_logging.py]8;;\:]8;id=5602689;file:///usr/local/lib/python3.12/dist-packages/sagemaker/core/telemetry/telemetry_logging.py#287\287]8;;\
                             understand our user's needs, diagnose issues, and deliver                             
                             additional features.                                                                  
                             To opt out of telemetry, please disable via TelemetryOptOut                           
                             parameter in SDK defaults config. For more information, refer                         
                             to                                                                                    
                             https://sagemaker.readthedocs.io/en/stable/overview.html#conf                         
                             iguring-and-using-defaults-with-the-sagemaker-python-sdk.                             

[07/17/26 03:03:50] INFO     Cannot simulate policies for                                  ]8;id=5602696;file:///usr/local/lib/python3.12/dist-packages/sagemaker/core/helper/iam_role_resolver.py\iam_role_resolver.py]8;;\:]8;id=5602697;file:///usr/local/lib/python3.12/dist-packages/sagemaker/core/helper/iam_role_resolver.py#363\363]8;;\
                             'arn:aws:iam::674622101542:role/NotebookTestEngine-Processing                         
                             JobRole' (access denied); permission verdict unknown.                                 

                    WARNING  Could not verify permissions for role                         ]8;id=5602703;file:///usr/local/lib/python3.12/dist-packages/sagemaker/core/helper/iam_role_resolver.py\iam_role_resolver.py]8;;\:]8;id=5602704;file:///usr/local/lib/python3.12/dist-packages/sagemaker/core/helper/iam_role_resolver.py#585\585]8;;\
                             'arn:aws:iam::674622101542:role/NotebookTestEngine-Processing                         
                             JobRole' (caller lacks iam:SimulatePrincipalPolicy).                                  
                             Proceeding with it. If the operation later fails with an                              
                             access-denied error, ensure the role has the required                                 
                             permissions for 'training' (see                                                       
                             IamRoleResolver().get_required_actions('training')) or create                         
                             a dedicated role via                                                                  
                             IamRoleResolver().create_execution_role(role_type='training')                         
                             .                                                                                     

                    INFO     Uploading InspectAI config to:                             ]8;id=5602711;file:///usr/local/lib/python3.12/dist-packages/sagemaker/train/evaluate/inspect_ai_evaluator.py\inspect_ai_evaluator.py]8;;\:]8;id=5602712;file:///usr/local/lib/python3.12/dist-packages/sagemaker/train/evaluate/inspect_ai_evaluator.py#657\657]8;;\
                             s3://notebook-test-engine-ds-674622101542-usw2/output/insp                            
                             ectai-eval/inspectai-config/49bf5f3f-b026-4c78-8adb-7579a0                            
                             bbb7ee/inspect_config.yaml                                                            

                    INFO     Resolved template parameters: {'job_name_prefix':                ]8;id=5602718;file:///usr/local/lib/python3.12/dist-packages/sagemaker/train/evaluate/base_evaluator.py\base_evaluator.py]8;;\:]8;id=5602719;file:///usr/local/lib/python3.12/dist-packages/sagemaker/train/evaluate/base_evaluator.py#915\915]8;;\
                             'eval-nova-1c30ebe7', 'image_uri':                                                    
                             '763104351884.dkr.ecr.us-west-2.amazonaws.com/sagemaker-inspect-                      
                             ai', 'role_arn':                                                                      
                             'arn:aws:iam::674622101542:role/NotebookTestEngine-ProcessingJob                      
                             Role', 'instance_type': 'ml.m5.large', 'max_runtime_seconds':                         
                             86400, 'config_s3_uri':                                                               
                             's3://notebook-test-engine-ds-674622101542-usw2/output/inspectai                      
                             -eval/inspectai-config/49bf5f3f-b026-4c78-8adb-7579a0bbb7ee',                         
                             's3_output_path':                                                                     
                             's3://notebook-test-engine-ds-674622101542-usw2/output/inspectai                      
                             -eval', 'kms_key_id': None, 'environment': None, 'vpc_config':                        
                             False}                                                                                

...

                    INFO     Updating pipeline resource.                                         ]8;id=5602754;file:///usr/local/lib/python3.12/dist-packages/sagemaker/core/resources.py\resources.py]8;;\:]8;id=5602755;file:///usr/local/lib/python3.12/dist-packages/sagemaker/core/resources.py#27371\27371]8;;\

                    INFO     Successfully updated pipeline:                                        ]8;id=5602761;file:///usr/local/lib/python3.12/dist-packages/sagemaker/train/evaluate/execution.py\execution.py]8;;\:]8;id=5602762;file:///usr/local/lib/python3.12/dist-packages/sagemaker/train/evaluate/execution.py#259\259]8;;\
                             SagemakerEvaluation-InspectAIEvaluation-1ce51c59-3ef3-4500-9c4c-001dd                 
                             b333879                                                                               

                    INFO     Starting pipeline execution: eval-nova-1c30ebe7-1784257431            ]8;id=5602768;file:///usr/local/lib/python3.12/dist-packages/sagemaker/train/evaluate/execution.py\execution.py]8;;\:]8;id=5602769;file:///usr/local/lib/python3.12/dist-packages/sagemaker/train/evaluate/execution.py#319\319]8;;\

[07/17/26 03:03:52] INFO     Pipeline execution started:                                           ]8;id=5602775;file:///usr/local/lib/python3.12/dist-packages/sagemaker/train/evaluate/execution.py\execution.py]8;;\:]8;id=5602776;file:///usr/local/lib/python3.12/dist-packages/sagemaker/train/evaluate/execution.py#330\330]8;;\
                             arn:aws:sagemaker:us-west-2:674622101542:pipeline/SagemakerEvaluation                 
                             -InspectAIEvaluation-1ce51c59-3ef3-4500-9c4c-001ddb333879/execution/y                 
                             yr2ie2hmt4s                                                                           


✓ Evaluation started!
  Execution ARN: arn:aws:sagemaker:us-west-2:674622101542:pipeline/SagemakerEvaluation-InspectAIEvaluation-1ce51c59-3ef3-4500-9c4c-001ddb333879/execution/yyr2ie2hmt4s
  Status: Executing


## Step 5: Monitor Progress

Use `refresh()` for manual status updates or `wait()` to block until completion.

In [12]:
# Check current status
execution.refresh()
pprint(execution.status)

[07/17/26 03:03:53] INFO     SageMaker Python SDK will collect telemetry to help us better ]8;id=5602781;file:///usr/local/lib/python3.12/dist-packages/sagemaker/core/telemetry/telemetry_logging.py\telemetry_logging.py]8;;\:]8;id=5602782;file:///usr/local/lib/python3.12/dist-packages/sagemaker/core/telemetry/telemetry_logging.py#287\287]8;;\
                             understand our user's needs, diagnose issues, and deliver                             
                             additional features.                                                                  
                             To opt out of telemetry, please disable via TelemetryOptOut                           
                             parameter in SDK defaults config. For more information, refer                         
                             to                                                                                    
                             https://sagemaker.readthedocs.io/en/stable/overview.html#conf                         
                             iguring-and-using-defaults-with-the-sagemaker-python-sdk.                             

/usr/local/lib/python3.12/dist-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


PipelineExecutionStatus(
│   overall_status='Executing',
│   step_details=[
│   │   StepDetail(
│   │   │   name='InspectAIEvaluation',
│   │   │   status='Starting',
│   │   │   start_time='2026-07-17T03:03:52.644000+00:00',
│   │   │   end_time=None,
│   │   │   display_name=None,
│   │   │   failure_reason=None,
│   │   │   job_arn=None
│   │   )
│   ],
│   failure_reason=None
)

In [13]:
# Wait for completion
execution.wait(target_status="Succeeded", poll=30, timeout=3600)

print(f"\nFinal Status: {execution.status.overall_status}")

[07/17/26 03:05:59] INFO     SageMaker Python SDK will collect telemetry to help us better ]8;id=5602855;file:///usr/local/lib/python3.12/dist-packages/sagemaker/core/telemetry/telemetry_logging.py\telemetry_logging.py]8;;\:]8;id=5602856;file:///usr/local/lib/python3.12/dist-packages/sagemaker/core/telemetry/telemetry_logging.py#287\287]8;;\
                             understand our user's needs, diagnose issues, and deliver                             
                             additional features.                                                                  
                             To opt out of telemetry, please disable via TelemetryOptOut                           
                             parameter in SDK defaults config. For more information, refer                         
                             to                                                                                    
                             https://sagemaker.readthedocs.io/en/stable/overview.html#conf                         
                             iguring-and-using-defaults-with-the-sagemaker-python-sdk.                             

╭─────────────────────────────────────────── Pipeline Execution Status ───────────────────────────────────────────╮
│  Evaluation Job        eval-nova-1c30ebe7-1784257431                                                            │
│  Links                 ]8;id=5602859;https://us-west-2.console.aws.amazon.com/cloudwatch/home?region=us-west-2#logsV2:log-groups/log-group/$252Faws$252Fsagemaker$252FTrainingJobs$3FlogStreamNameFilter$3Dpipelines-yyr2ie2hmt4s-InspectAIEvaluation-PcovPTJNCp\🔗 CloudWatch Logs]8;;\                                                                       │
│  Overall Status        Succeeded                                                                                │
│  Target Status         Succeeded                                                                                │
│  Elapsed Time          124.6s                                                                                   │
│                                                                                                                 │
│ Pipeline Steps                                                                                                  │
│  Step Name                       Status           Duration                                                      │
│  InspectAIEvaluation             Succeeded        120.5s                                                        │
│                                                                                                                 │
│ Job ARNs                                                                                                        │
│  Step                  Console  Logs     Job ARN                                                                │
│  InspectAIEvaluation   ]8;id=5602862;https://us-west-2.console.aws.amazon.com/sagemaker/home?region=us-west-2#/jobs/pipelines-yyr2ie2hmt4s-InspectAIEvaluation-PcovPTJNCp\🔗 link]8;;\  ]8;id=5602865;https://us-west-2.console.aws.amazon.com/cloudwatch/home?region=us-west-2#logsV2:log-groups/log-group/$252Faws$252Fsagemaker$252FTrainingJobs$3FlogStreamNameFilter$3Dpipelines-yyr2ie2hmt4s-InspectAIEvaluation-PcovPTJNCp\🔗 link]8;;\  arn:aws:sagemaker:us-west-2:674622101542:training-job/pipelines-yyr2i  │
│                                          e2hmt4s-InspectAIEvaluation-PcovPTJNCp                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

                    INFO     Final Resource Status: Succeeded                                     ]8;id=5602871;file:///usr/local/lib/python3.12/dist-packages/sagemaker/train/evaluate/execution.py\execution.py]8;;\:]8;id=5602872;file:///usr/local/lib/python3.12/dist-packages/sagemaker/train/evaluate/execution.py#1310\1310]8;;\


Final Status: Succeeded


## Step 6: View Results

In [14]:
execution.show_results()

[07/17/26 03:06:00] INFO     SageMaker Python SDK will collect telemetry to help us better ]8;id=5602877;file:///usr/local/lib/python3.12/dist-packages/sagemaker/core/telemetry/telemetry_logging.py\telemetry_logging.py]8;;\:]8;id=5602878;file:///usr/local/lib/python3.12/dist-packages/sagemaker/core/telemetry/telemetry_logging.py#287\287]8;;\
                             understand our user's needs, diagnose issues, and deliver                             
                             additional features.                                                                  
                             To opt out of telemetry, please disable via TelemetryOptOut                           
                             parameter in SDK defaults config. For more information, refer                         
                             to                                                                                    
                             https://sagemaker.readthedocs.io/en/stable/overview.html#conf                         
                             iguring-and-using-defaults-with-the-sagemaker-python-sdk.                             

[07/17/26 03:06:01] INFO     SageMaker Python SDK will collect telemetry to help us better ]8;id=5602883;file:///usr/local/lib/python3.12/dist-packages/sagemaker/core/telemetry/telemetry_logging.py\telemetry_logging.py]8;;\:]8;id=5602884;file:///usr/local/lib/python3.12/dist-packages/sagemaker/core/telemetry/telemetry_logging.py#287\287]8;;\
                             understand our user's needs, diagnose issues, and deliver                             
                             additional features.                                                                  
                             To opt out of telemetry, please disable via TelemetryOptOut                           
                             parameter in SDK defaults config. For more information, refer                         
                             to                                                                                    
                             https://sagemaker.readthedocs.io/en/stable/overview.html#conf                         
                             iguring-and-using-defaults-with-the-sagemaker-python-sdk.                             

                    INFO     Extracted training job name:                                  ]8;id=5602891;file:///usr/local/lib/python3.12/dist-packages/sagemaker/train/common_utils/show_results_utils.py\show_results_utils.py]8;;\:]8;id=5602892;file:///usr/local/lib/python3.12/dist-packages/sagemaker/train/common_utils/show_results_utils.py#66\66]8;;\
                             pipelines-yyr2ie2hmt4s-InspectAIEvaluation-PcovPTJNCp from                            
                             step: InspectAIEvaluation                                                             

InspectAI Evaluation Results

══════════════════════════════════════════════════════════════════════

┏━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ Property        ┃ Value                                                                                         ┃
┡━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│ Training Job    │ pipelines-yyr2ie2hmt4s-InspectAIEvaluation-PcovPTJNCp                                         │
│ Status          │ Completed                                                                                     │
│ S3 Output       │ s3://notebook-test-engine-ds-674622101542-usw2/output/inspectai-eval                          │
│ Model Artifacts │ s3://notebook-test-engine-ds-674622101542-usw2/output/inspectai-eval/pipelines-yyr2ie2hmt4s-… │
│ Duration        │ 0:01:18.680000                                                                                │
└─────────────────┴───────────────────────────────────────────────────────────────────────────────────────────────┘

══════════════════════════════════════════════════════════════════════

Full InspectAI evaluation logs available at:
  s3://notebook-test-engine-ds-674622101542-usw2/output/inspectai-eval/pipelines-yyr2ie2hmt4s-InspectAIEvaluation-P
covPTJNCp/output/

## Retrieve an Existing Evaluation

You can retrieve a previously started evaluation job using its ARN.

In [15]:
from sagemaker.train.evaluate import EvaluationPipelineExecution

# Retrieve by ARN
existing_arn = execution.arn  # Or paste a specific ARN
existing_exec = EvaluationPipelineExecution.get(arn=existing_arn, region=REGION)

print(f"Retrieved: {existing_exec.name}")
print(f"Status: {existing_exec.status.overall_status}")

[07/17/26 03:06:02] INFO     SageMaker Python SDK will collect telemetry to help us better ]8;id=5602897;file:///usr/local/lib/python3.12/dist-packages/sagemaker/core/telemetry/telemetry_logging.py\telemetry_logging.py]8;;\:]8;id=5602898;file:///usr/local/lib/python3.12/dist-packages/sagemaker/core/telemetry/telemetry_logging.py#287\287]8;;\
                             understand our user's needs, diagnose issues, and deliver                             
                             additional features.                                                                  
                             To opt out of telemetry, please disable via TelemetryOptOut                           
                             parameter in SDK defaults config. For more information, refer                         
                             to                                                                                    
                             https://sagemaker.readthedocs.io/en/stable/overview.html#conf                         
                             iguring-and-using-defaults-with-the-sagemaker-python-sdk.                             

[07/17/26 03:06:03] INFO     Extracted s3_output_path from training job                            ]8;id=5602904;file:///usr/local/lib/python3.12/dist-packages/sagemaker/train/evaluate/execution.py\execution.py]8;;\:]8;id=5602905;file:///usr/local/lib/python3.12/dist-packages/sagemaker/train/evaluate/execution.py#430\430]8;;\
                             pipelines-yyr2ie2hmt4s-InspectAIEvaluation-PcovPTJNCp:                                
                             s3://notebook-test-engine-ds-674622101542-usw2/output/inspectai-eval                  

Retrieved: yyr2ie2hmt4s
Status: Succeeded


## List All InspectAI Evaluations

In [16]:
all_executions = list(InspectAIEvaluator.get_all(region=REGION))

print(f"Found {len(all_executions)} InspectAI evaluation(s):\n")
for exec in all_executions[:10]:
    print(f"  - {exec.name}: {exec.status.overall_status}")

                    INFO     SageMaker Python SDK will collect telemetry to help us better ]8;id=5602910;file:///usr/local/lib/python3.12/dist-packages/sagemaker/core/telemetry/telemetry_logging.py\telemetry_logging.py]8;;\:]8;id=5602911;file:///usr/local/lib/python3.12/dist-packages/sagemaker/core/telemetry/telemetry_logging.py#287\287]8;;\
                             understand our user's needs, diagnose issues, and deliver                             
                             additional features.                                                                  
                             To opt out of telemetry, please disable via TelemetryOptOut                           
                             parameter in SDK defaults config. For more information, refer                         
                             to                                                                                    
                             https://sagemaker.readthedocs.io/en/stable/overview.html#conf                         
                             iguring-and-using-defaults-with-the-sagemaker-python-sdk.                             

[07/17/26 03:06:05] INFO     SageMaker Python SDK will collect telemetry to help us better ]8;id=5602916;file:///usr/local/lib/python3.12/dist-packages/sagemaker/core/telemetry/telemetry_logging.py\telemetry_logging.py]8;;\:]8;id=5602917;file:///usr/local/lib/python3.12/dist-packages/sagemaker/core/telemetry/telemetry_logging.py#287\287]8;;\
                             understand our user's needs, diagnose issues, and deliver                             
                             additional features.                                                                  
                             To opt out of telemetry, please disable via TelemetryOptOut                           
                             parameter in SDK defaults config. For more information, refer                         
                             to                                                                                    
                             https://sagemaker.readthedocs.io/en/stable/overview.html#conf                         
                             iguring-and-using-defaults-with-the-sagemaker-python-sdk.                             

                    INFO     Extracted s3_output_path from training job                            ]8;id=5602922;file:///usr/local/lib/python3.12/dist-packages/sagemaker/train/evaluate/execution.py\execution.py]8;;\:]8;id=5602923;file:///usr/local/lib/python3.12/dist-packages/sagemaker/train/evaluate/execution.py#430\430]8;;\
                             pipelines-yyr2ie2hmt4s-InspectAIEvaluation-PcovPTJNCp:                                
                             s3://notebook-test-engine-ds-674622101542-usw2/output/inspectai-eval                  

[07/17/26 03:06:06] INFO     Extracted s3_output_path from training job                            ]8;id=5602928;file:///usr/local/lib/python3.12/dist-packages/sagemaker/train/evaluate/execution.py\execution.py]8;;\:]8;id=5602929;file:///usr/local/lib/python3.12/dist-packages/sagemaker/train/evaluate/execution.py#430\430]8;;\
                             pipelines-tgnsps5el9ej-InspectAIEvaluation-YKnm7xfFHt:                                
                             s3://notebook-test-engine-ds-674622101542-usw2/inspectai/eval-output                  

Found 2 InspectAI evaluation(s):

  - yyr2ie2hmt4s: Succeeded
  - tgnsps5el9ej: Succeeded


## Stop a Running Job (Optional)

In [17]:
# Uncomment to stop the job
# execution.stop()
# print(f"Execution stopped. Status: {execution.status.overall_status}")

## Summary

### Inference Modes
| Mode | Parameters | Use Case |
|------|-----------|----------|
| Bedrock | `bedrock_model_id` | Easiest — no endpoint management |
| Existing endpoint | `endpoint_name` | Re-use a running endpoint |
| Create endpoint | `model_s3_uri` + `inference_image_uri` | Custom models not on Bedrock |

### Key Parameters
- `benchmarks_path`: S3 URI to your InspectAI `.py` task files
- `tasks`: List of task configurations (name, limit, epochs)
- `temperature`, `top_p`, `top_k`, `max_tokens`: Decoding tunables
- `max_connections`: Concurrent inference connections (default 16)
- `instance_type`: Orchestrator instance type (default `ml.m5.large`)